# Module 1: Concepts Walkthrough

Welcome to the **ParcelFlow** concepts walkthrough. In this notebook, we will explore the internal components of a Data-Driven Workflow Engine.

**Learning Objectives:**
1. Understand the `Parcel` (data) vs `Node` (logic) separation.
2. Observe how the `WorkflowEngine` discovers executable work.
3. See the difference between explicit DAGs and implicit data dependencies.

In [ ]:
import sys
import os

# Add the parent directory to the path so we can import the engine
sys.path.append(os.path.abspath('..'))

from workflow_engine import WorkflowEngine
from base_node import BaseNode
import logging

# Setup logging to see what's happening
logging.basicConfig(level=logging.INFO, format='%(message)s')
print("ParcelFlow environment loaded!")

## 1. The Atomic Unit: The Node

In ParcelFlow, a **Node** is a self-contained unit of logic. Crucially, a node **does not know** what runs before it or after it. It only knows:
1. What data it needs (`requires`)
2. What data it produces (`outputs`)

Let's define a simple node that adds two numbers.

In [ ]:
class AdderNode(BaseNode):
    def __init__(self):
        # I need 'a' and 'b'. I create 'sum'.
        super().__init__(requires=["a", "b"], outputs=["sum"])

    def run(self, parcels, index=None):
        val_a = parcels["a"].value
        val_b = parcels["b"].value
        
        result = val_a + val_b
        
        print(f"[AdderNode] calculated {val_a} + {val_b} = {result}")
        return {"sum": result}

## 2. The Engine Loop

The `WorkflowEngine` is the "brain". It doesn't have a plan. It just looks at the state of the world (the `parcels`) and asks: *"Who can run right now?"*

Let's give it some data and our node.

In [ ]:
# Initialize Engine
engine = WorkflowEngine()

# Define available Nodes
nodes = [AdderNode()]

# 1. Try running with NO data
print("--- Attempt 1: No Data ---")
engine.execute_workflow(nodes, initial_data={})
print("(Notice nothing happened. The node is waiting for 'a' and 'b')")

In [ ]:
# 2. Try running with PARTIAL data
print("\n--- Attempt 2: Partial Data ---")
engine.execute_workflow(nodes, initial_data={"a": 10})
print("(Still nothing. Need 'b')")

In [ ]:
# 3. Run with FULL data
print("\n--- Attempt 3: Full Data ---")
result = engine.execute_workflow(nodes, initial_data={"a": 10, "b": 5})
print("\nFinal Result:", result['sum'].value)

## 3. Implicit Chaining

Now let's add a second node that needs the output of the first node. 

We will add a `MultiplierNode` that requires `sum` (which `AdderNode` produces) and a new value `c`.

In [ ]:
class MultiplierNode(BaseNode):
    def __init__(self):
        # Dependencies: 'sum' comes from AdderNode, 'c' comes from input
        super().__init__(requires=["sum", "c"], outputs=["final_product"])

    def run(self, parcels, index=None):
        val_sum = parcels["sum"].value
        val_c = parcels["c"].value
        
        result = val_sum * val_c
        print(f"[MultiplierNode] calculated {val_sum} * {val_c} = {result}")
        return {"final_product": result}

# Create the pipeline
pipeline = [AdderNode(), MultiplierNode()]

# Execute
result = engine.execute_workflow(pipeline, initial_data={"a": 2, "b": 3, "c": 4})
print("\nFinal Product:", result.get('final_product').value)

### Observation

Notice we never told the engine "Run Adder then Multiplier". 

1. Engine sees `a`, `b`, `c` exist.
2. `AdderNode` needs `a, b`. **Ready.**
3. `MultiplierNode` needs `sum, c`. `sum` is missing. **Not Ready.**
4. `AdderNode` runs -> produces `sum`.
5. Now `MultiplierNode` has `sum, c`. **Ready.**
6. `MultiplierNode` runs.

This is **Data-Driven Execution**.